# Smart Home Energy Manager — Topic 12 walkthrough notebook
**AIF1303 AI Fundamentals · Final Project · Tuleu Tamerlan, Esen Nuradil (stream 1, group 3)**

This notebook reproduces every number of the project with the *same* Python functions the web app uses (`backend/algorithms`).
Each method is shown as **input → key calculation → output → one-sentence explanation**, as the course brief asks.

All inputs are **stated assumptions** (toy model, no hardware, no real dataset). Run the cells top to bottom; only the Python standard library is needed.
To run in Google Colab: upload the project folder (or clone the GitHub repository) and set the working directory to `notebooks/`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # the project root, so `backend` can be imported
from backend import brief
from backend.algorithms.bayes import bayes_details, predict_occupancy, filter_sequence
from backend.algorithms.mdp import interval_costs, one_step_decision, qmdp_decision, value_iteration
from backend.algorithms.hill_climbing import optimise_schedule, exact_optimum
print("Python", sys.version.split()[0], "- imports OK")

Python 3.12.8 - imports OK


## 0. Problem formulation (RO1)
| Element | Topic 12 |
|---|---|
| Agent | controller of one appliance (48 W fan) |
| Percept | one noisy motion reading per 15-minute interval |
| Hidden state | true occupancy: occupied / empty |
| Actions | Keep On / Turn Off |
| Utility | −(electricity cost) − (discomfort cost if someone is present and the fan is off) − (start-up cost) |
| Environment | partially observable, stochastic, sequential, single agent |

In [2]:
a = brief.run()["assumptions"]
for key in ("device", "intervals", "interval_minutes", "tariff", "comfort_per_hour", "startup_cost",
            "motion_if_occupied", "motion_if_empty", "stay_occupied", "become_occupied", "priors", "readings", "truth"):
    print(f"{key:>20}: {a[key]}")
print("cost of one ON interval   :", round(a["costs"]["energy"], 3), "KZT")
print("discomfort per OFF interval if occupied:", round(a["costs"]["comfort"], 3), "KZT")

              device: {'name': 'Living-room fan', 'power_w': 48}
           intervals: 8
    interval_minutes: 15
              tariff: 25.0
    comfort_per_hour: 3.0
        startup_cost: 0.3
  motion_if_occupied: 0.9
     motion_if_empty: 0.2
       stay_occupied: 0.85
     become_occupied: 0.1
              priors: [0.1, 0.2, 0.75, 0.75, 0.75, 0.55, 0.4, 0.15]
            readings: [False, False, True, True, False, False, False, False]
               truth: [False, False, True, True, True, False, False, False]
cost of one ON interval   : 0.3 KZT
discomfort per OFF interval if occupied: 0.75 KZT


## 1. Hill climbing — the lowest-cost of 5 operating schedules (local search)
* **State:** one of 5 candidate schedules, ordered narrow → broad.
* **Objective (minimise):** `Σ ON intervals × energy + Σ OFF intervals × P(occupied) × discomfort + start-up × starts`.
* **Neighbours:** adjacent candidates (one interval added or removed).
* **Move:** to the cheaper neighbour only if it is strictly cheaper. **Stop:** no cheaper neighbour.

In [3]:
for row in brief.candidates():
    print(f"schedule {row['number']} {row['times']:<12} energy {row['energy']:.4f} + discomfort {row['discomfort']:.4f}"
          f" + start-up {row['startup']:.4f} = {row['total']:.4f}")
hc = brief.hill_climb(start=1)
for step in hc["steps"]:
    print("  at", step["at"], "->", step["explanation"])
print("OUTPUT: best schedule", hc["best"], "| path", hc["path"], "| exhaustive check:", hc["exhaustive_best"])

schedule 1 18:45-19:00  energy 0.3000 + discomfort 2.1750 + start-up 0.3000 = 2.7750
schedule 2 18:30-19:00  energy 0.6000 + discomfort 1.6125 + start-up 0.3000 = 2.5125
schedule 3 18:30-19:15  energy 0.9000 + discomfort 1.0500 + start-up 0.3000 = 2.2500
schedule 4 18:30-19:30  energy 1.2000 + discomfort 0.6375 + start-up 0.3000 = 2.1375
schedule 5 18:15-19:30  energy 1.5000 + discomfort 0.4875 + start-up 0.3000 = 2.2875
  at 1 -> Schedule 2 costs 2.5125 < 2.7750: move.
  at 2 -> Schedule 3 costs 2.2500 < 2.5125: move.
  at 3 -> Schedule 4 costs 2.1375 < 2.2500: move.
  at 4 -> No neighbour is cheaper than 2.1375: local optimum, stop.
OUTPUT: best schedule 4 | path [1, 2, 3, 4] | exhaustive check: 4


**Explanation:** each step keeps a strictly cheaper neighbour, so the search stops at schedule 4 (18:30–19:30), where adding or removing one interval no longer lowers the expected cost. Exhaustive checking of all 5 candidates confirms it is the global optimum here.

Hill climbing *can* get stuck. With a different occupancy profile it stops at schedule 1 although schedule 5 is cheaper; a restart from another candidate escapes:

In [4]:
trap = (0.0, 1.0, 0.0, 0.5, 1.0, 1.0, 0.0, 0.0)
print("start 1 :", brief.hill_climb(1, trap)["path"], "global optimum:", brief.hill_climb(1, trap)["exhaustive_best"])
print("start 3 :", brief.hill_climb(3, trap)["path"])

start 1 : [1] global optimum: 5
start 3 : [3, 4, 5]


## 2. Bayesian inference — is the room occupied? (probabilistic reasoning)
`P(occ | reading) = P(reading | occ) · P(occ) / P(reading)`, with `P(reading) = P(reading | occ) P(occ) + P(reading | empty) P(empty)`.
One prior and one sensor observation, then one step through the transition table.

In [5]:
b = brief.bayes_step(focus=5)
print(f"input : interval {b['interval']} ({b['time']}), prior {b['prior']}, reading = {'motion' if b['motion'] else 'no motion'}")
print(f"P(reading)        = {b['likelihood_occupied']:.2f} x {b['prior']:.2f} + {b['likelihood_empty']:.2f} x {1-b['prior']:.2f} = {b['evidence']:.4f}")
print(f"P(occ | reading)  = {b['likelihood_occupied']*b['prior']:.4f} / {b['evidence']:.4f} = {b['posterior']:.4f}")
print(f"P(occ next)       = {b['posterior']:.4f} x 0.85 + {1-b['posterior']:.4f} x 0.10 = {b['predicted']:.4f}")
m = brief.bayes_step(focus=3)
print(f"with motion at interval 3 (prior 0.75): posterior {m['posterior']:.4f}")

input : interval 5 (19:00-19:15), prior 0.75, reading = no motion
P(reading)        = 0.10 x 0.75 + 0.80 x 0.25 = 0.2750
P(occ | reading)  = 0.0750 / 0.2750 = 0.2727
P(occ next)       = 0.2727 x 0.85 + 0.7273 x 0.10 = 0.3045
with motion at interval 3 (prior 0.75): posterior 0.9310


**Explanation:** the sensor misses a still person 10 % of the time, so “no motion” lowers the belief from 0.75 to 0.273 but not to zero; motion raises it to 0.931.

The full application chains readings: the HMM forward algorithm (predict → update) over consecutive intervals:

In [6]:
rows = filter_sequence(0.5, [True, True, False, False, False], 0.9, 0.2, 0.85, 0.10)
for k, r in enumerate(rows, 1):
    print(f"t={k}: prior {r['prior']:.3f} -> posterior {r['posterior']:.3f} -> prediction {r['predicted']:.3f}")

t=1: prior 0.500 -> posterior 0.818 -> prediction 0.714
t=2: prior 0.714 -> posterior 0.918 -> prediction 0.789
t=3: prior 0.789 -> posterior 0.318 -> prediction 0.339
t=4: prior 0.339 -> posterior 0.060 -> prediction 0.145
t=5: prior 0.145 -> posterior 0.021 -> prediction 0.116


## 3. MDP — Keep On or Turn Off (sequential decision making)
States: occupied / empty. Actions: Keep On / Turn Off. Rewards are negative costs. One-step expected reward (one Bellman update with V_next = 0):
`Q(a) = P(occ) · R(occ, a) + P(empty) · R(empty, a)`; the larger Q wins.

In [7]:
mdp = brief.mdp_step(b["predicted"], was_on=True)
print("rewards     :", mdp["rewards"])
print("transitions :", mdp["transitions"])
print(f"Q(Keep On) = {mdp['q']['Keep On']:.4f}   Q(Turn Off) = {mdp['q']['Turn Off']:.4f}   break-even P = {mdp['break_even']:.2f}")
print("OUTPUT action:", mdp["action"])
p = mdp["planner"]
print(f"application planner (value iteration, H=4, start-up cost): Keep On {p['q']['Keep On']:.4f} vs Turn Off {p['q']['Turn Off']:.4f} -> {p['action']}")

rewards     : {'Keep On': {'occupied': -0.3, 'empty': -0.3}, 'Turn Off': {'occupied': -0.75, 'empty': 0.0}}
transitions : {'occupied': {'occupied': 0.85, 'empty': 0.15000000000000002}, 'empty': {'occupied': 0.1, 'empty': 0.9}}
Q(Keep On) = -0.3000   Q(Turn Off) = -0.2284   break-even P = 0.40
OUTPUT action: Turn Off
application planner (value iteration, H=4, start-up cost): Keep On -0.6501 vs Turn Off -0.6770 -> Keep On


**Explanation:** P(occupied) = 0.305 is below the break-even 0.40, so one step ahead the expected discomfort (0.228 KZT) is cheaper than the electricity (0.300 KZT) → *Turn Off*. The application's planner looks 4 intervals ahead and includes the start-up cost: switching off now would probably mean paying it again soon → *Keep On*. This is why the app uses value iteration rather than the one-step rule.

Value iteration agrees with brute-force expectimax (also covered by a unit test):

In [8]:
costs = interval_costs(48, 15, 25.0, 3.0, 0.3)
q0, values = value_iteration([("ON", "OFF")] * 3, costs, 0.85, 0.10)
for state, q in q0.items():
    print("state (occupied, was_on) =", state, {k: round(v, 4) for k, v in q.items()})

state (occupied, was_on) = (1, True) {'ON': -0.7807, 'OFF': -1.4857}
state (occupied, was_on) = (1, False) {'ON': -1.0807, 'OFF': -1.4857}
state (occupied, was_on) = (0, True) {'ON': -0.4095, 'OFF': -0.1395}
state (occupied, was_on) = (0, False) {'ON': -0.7095, 'OFF': -0.1395}


## 4. Results — the same evening three ways
Same readings and the chosen schedule; the true occupancy is used **only for scoring**.

In [9]:
ev = brief.run()["evaluation"]
labels = {"schedule": "Schedule only", "one_step": "AI one-step MDP", "planner": "AI planner H=4"}
for key, t in ev["totals"].items():
    print(f"{labels[key]:<16} energy {t['wh']:5.1f} Wh  starts {t['starts']}  occupied-but-off {t['missed_min']:3d} min"
          f"  on-while-empty {t['wasted_min']:3d} min  total cost {t['cost']:.3f} KZT")
print()
for r in ev["rows"]:
    print(f"{r['time']}  motion={'Y' if r['motion'] else '-'} truth={'O' if r['truth'] else '-'}  P(occ)={r['belief']:.3f}"
          f"  schedule={r['schedule_action']:<3} one-step={r['one_step_action']:<3} planner={r['planner_action']}")

Schedule only    energy  48.0 Wh  starts 1  occupied-but-off   0 min  on-while-empty  15 min  total cost 1.500 KZT
AI one-step MDP  energy  24.0 Wh  starts 1  occupied-but-off  15 min  on-while-empty   0 min  total cost 1.650 KZT
AI planner H=4   energy  36.0 Wh  starts 1  occupied-but-off   0 min  on-while-empty   0 min  total cost 1.200 KZT

18:00-18:15  motion=- truth=-  P(occ)=0.110  schedule=OFF one-step=OFF planner=OFF
18:15-18:30  motion=- truth=-  P(occ)=0.123  schedule=OFF one-step=OFF planner=OFF
18:30-18:45  motion=Y truth=O  P(occ)=0.798  schedule=ON  one-step=ON  planner=ON
18:45-19:00  motion=Y truth=O  P(occ)=0.798  schedule=ON  one-step=ON  planner=ON
19:00-19:15  motion=- truth=O  P(occ)=0.305  schedule=ON  one-step=OFF planner=ON
19:15-19:30  motion=- truth=-  P(occ)=0.199  schedule=ON  one-step=OFF planner=OFF
19:30-19:45  motion=- truth=-  P(occ)=0.158  schedule=OFF one-step=OFF planner=OFF
19:45-20:00  motion=- truth=-  P(occ)=0.116  schedule=OFF one-step=OFF plann

**Interpretation:** the planner saves 25 % of the schedule's energy with no missed comfort; the one-step rule saves 50 % energy but leaves a present person without the fan for 15 minutes. Energy and comfort are reported side by side because the exchange rate between them is a user assumption (comfort value).

## 5. Scaling up: the full application on the 7-day synthetic demo
The web app applies the same methods to 3 rooms and 7 devices with 24-hour schedules (seeded synthetic data, not measurements).

In [10]:
import tempfile
os.environ["SHEM_DATA_DIR"] = tempfile.mkdtemp(prefix="shem-nb-")
from backend import analytics, db, demo
info = demo.generate_demo()
conn = db.connect("demo")
for row in analytics.compare_strategies(conn)["strategies"]:
    print(f"{row['label']:<34} {row['kwh']:8.2f} kWh  starts {row['starts']:4d}  missed {row['missed_min']:5d} min  on-while-empty {row['wasted_min']:5d} min")
conn.close()

Schedule only                        226.73 kWh  starts   57  missed     0 min  on-while-empty 11460 min
AI: one-step MDP rule                145.57 kWh  starts  198  missed  1785 min  on-while-empty  1425 min
AI: value iteration, H = 4           143.66 kWh  starts  163  missed  1845 min  on-while-empty  1440 min
Oracle (knows true occupancy)        156.55 kWh  starts   83  missed     0 min  on-while-empty     0 min


In [11]:
# 24-hour schedule search: hill climbing with random restarts vs the exact optimum (dynamic programming)
priors = [0.05] * 24
for hour in list(range(7, 9)) + list(range(18, 23)):
    priors[hour] = 0.95
start = [1 if 7 <= h < 9 else 0 for h in range(24)]
for restarts in (0, 10):
    r = optimise_schedule(priors, 2.0, 6.0, 9.0, start, restarts=restarts, seed=1)
    print(f"restarts={restarts:2d}: hill climbing {r['best']['total']:.2f}  exact {r['exact']['total']:.2f}  gap {r['gap_to_exact']:.2f}")

restarts= 0: hill climbing 46.60  exact 35.50  gap 11.10
restarts=10: hill climbing 35.50  exact 35.50  gap 0.00


## 6. Automated tests
`python -m unittest -v` in the project root runs the whole suite (algorithms, scenario, CSV import, HTTP API, hosting entry point, and this brief).

In [12]:
import unittest, io
suite = unittest.defaultTestLoader.discover(os.path.abspath(".."), pattern="test_*.py", top_level_dir=os.path.abspath(".."))
result = unittest.TextTestRunner(stream=io.StringIO(), verbosity=0).run(suite)
print("tests run:", result.testsRun, "| failures:", len(result.failures), "| errors:", len(result.errors))

tests run: 64 | failures: 0 | errors: 0


## Limitations
No hardware (virtual devices, energy = rated power × time); synthetic or user-entered data only; one sensor per room; QMDP is an approximation of the POMDP; comfort values and start-up costs are user assumptions. See `REPORT.md`.